In [ ]:
input_file = "owning_agg.json"

In [ ]:
import matplotlib.pyplot as plt

# Ensure plots render inline inside Jupyter
%matplotlib inline 

import json 

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

In [ ]:
with open(input_file, encoding="utf-8") as f:
    agg_data = json.load(f)

initial_nav = agg_data["initial_nav"]
years = agg_data["years_to_simulate"]
num_paths = agg_data["total_paths"]
retirement_age = agg_data["retirement_age"]

display(Markdown("# Analysis Results"))
display(Markdown("## Simulation Parameters"))
display(Markdown(f"""| Initial NAV | Age of Retirement | Years to Simulate | Total Paths |
| :--- | :--- | :--- | :--- |
| ${initial_nav:,.2f} | {retirement_age} | {years} | {num_paths} |"""))

display(Markdown("## Results per model"))

for m in agg_data["results"]:
    data = agg_data["results"][m]
    df_data = pd.DataFrame(data)
    display(Markdown(f"## {m}"))
    df_data = df_data.sort_values(by=['spending', 'allocation', 'tax_regime'], ascending=[True, True, True])
    df_data["ruin_rate"] = df_data["ruin_path_count"] * 100.0 / num_paths
    df_data["min_ruin_age"] = df_data["ruin_month_min"] / 12.0 + retirement_age
    df_data["ruin_age_p50"] = df_data["ruin_month_median"] / 12.0 + retirement_age
    df_data["ruin_age_es5"] = df_data["ruin_month_es5"] / 12.0 + retirement_age
    df_data["ruin_age_es10"] = df_data["ruin_month_es10"] / 12.0 + retirement_age
    df_data["p5_return"] *= 100.0
    df_data["p10_return"] *= 100.0
    df_data = df_data.drop(columns=[
        "ruin_path_count", "ruin_month_min", "ruin_month_median",
        "ruin_month_es5", "ruin_month_es10"])
    df_data = df_data.style.format_index(lambda x: str(x).replace("_", " ").title(), axis=1)
    df_data = df_data.format({
        "spending": "${:,.0f}",
        "ruin_rate": "{:.1f}%",
        "min_ruin_age": "{:.1f}y",
        "ruin_age_p50": "{:.1f}y",
        "ruin_age_es5": "{:.1f}y",
        "ruin_age_es10": "{:.1f}y",
        "p5_return": "{:.2f}%",
        "p10_return": "{:.2f}%",
        "p25_return": "{:.2f}x",
        "p50_return": "{:.2f}x"},
        na_rep="N/A")
    display(df_data)